In [61]:
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, Boolean, ForeignKey, DateTime
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
import psycopg2
import re

## Summaries

In [62]:
# Step 1: Load the CSV data
df = pd.read_csv('output_with_documents.csv')

# Display basic info about the data
print(f"DataFrame shape: {df.shape}")
print("\nColumn info:")
df.info()

print("\nSample data:")
df.head()

DataFrame shape: (431, 6)

Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 431 entries, 0 to 430
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Title                      431 non-null    object
 1   Link                       431 non-null    object
 2   Date                       431 non-null    object
 3   federal_register_link      124 non-null    object
 4   contains_federal_register  431 non-null    bool  
 5   document_id                62 non-null     object
dtypes: bool(1), object(5)
memory usage: 17.4+ KB

Sample data:


,Title,Link,Date,federal_register_link,contains_federal_register,document_id
0,Rollback: DOJ Directed to “Narrow” Use of Disp...,https://eelp.law.harvard.edu/tracker/rollback-...,"February 5, 2025",NaN,False,NaN
1,Rollback: DOJ Terminated First-Ever Office of ...,https://eelp.law.harvard.edu/tracker/doj-launc...,"February 4, 2025",https://www.federalregister.gov/documents/2021...,True,NaN
2,Rollback: FEMA Disbanded National Advisory Cou...,https://eelp.law.harvard.edu/tracker/fema-anno...,"January 24, 2025",NaN,False,NaN
3,Rollback: CEQ’s Climate & Economic Justice Scr...,https://eelp.law.harvard.edu/tracker/ceqs-clim...,"January 22, 2025",https://www.federalregister.gov/documents/2021...,True,NaN
4,Rollback: Trump Rescinded Biden’s Executive Or...,https://eelp.law.harvard.edu/tracker/president...,"January 22, 2025",https://www.federalregister.gov/documents/2023...,True,NaN


In [63]:
# Step 2: Create database connection
# Replace these with your PostgreSQL credentials
username = "postgres"  # Your PostgreSQL username
password = "your_password"  # Your PostgreSQL password, if any
host = "localhost"
port = "5432"
database_name = "ycej_database"

# Create a connection to PostgreSQL server (without specifying a database)
engine_server = create_engine(f'postgresql://{username}:{password}@{host}:{port}/postgres')

# Check if our database exists, and create it if it doesn't
with engine_server.connect() as conn:
    # Use text() for SQL execution in newer SQLAlchemy versions
    from sqlalchemy import text
    conn.execute(text("commit"))  # End the transaction
    
    # Check if database exists
    result = conn.execute(text(f"SELECT 1 FROM pg_database WHERE datname = '{database_name}'"))
    exists = result.scalar()
    if not exists:
        print(f"Creating database: {database_name}")
        conn.execute(text(f"CREATE DATABASE {database_name}"))
    else:
        print(f"Database '{database_name}' already exists")

# Connect to our database
engine = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{database_name}')

# Step 3: Define the database schema

Database 'ycej_database' already exists


In [64]:

# Step 3: Define the database schema
Base = declarative_base()

class Document(Base):
    __tablename__ = 'summaries'
    
    id = Column(Integer, primary_key=True)
    title = Column(String)
    link = Column(String)
    date = Column(String)  # Consider using DateTime if dates are standardized
    federal_register_link = Column(String)
    contains_federal_register = Column(Boolean)
    document_id = Column(String, unique=True)  # This will be our key for relating to other tables
    
    def __repr__(self):
        return f"<Document(document_id='{self.document_id}', title='{self.title}')>"

/var/folders/2z/q_7y7r7525qcys0tvfht24m00000gp/T/ipykernel_63515/1724944353.py:2: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [65]:

# Step 4: Create the tables
Base.metadata.create_all(engine)


In [66]:

# Step 5: Insert data into the tables
# Clean the DataFrame
# Remove any problematic characters from string columns
string_columns = df.select_dtypes(include=['object']).columns
for col in string_columns:
    # Replace NaN values with empty strings for most columns
    # but leave document_id as None/NULL if it's empty
    if col != 'document_id':
        df[col] = df[col].fillna('')

# Convert column names to lowercase to match our model
df.columns = [col.lower() for col in df.columns]

# Convert the DataFrame to a list of dictionaries
data_to_insert = df.to_dict(orient='records')

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

try:
    # Get existing document IDs to avoid duplicates (only for non-null IDs)
    existing_doc_ids = set(doc_id[0] for doc_id in session.query(Document.document_id).all() 
                          if doc_id[0] is not None)
    
    # Insert records, handling duplicates appropriately
    new_records = 0
    skipped_records = 0
    null_id_records = 0
    
    for record in data_to_insert:
        try:
            # If document_id is empty/null, we can insert it
            # If document_id exists but isn't in our database yet, we can insert it
            # If document_id exists and is already in our database, we'll skip it
            
            doc_id = record.get('document_id')
            
            if doc_id is None or doc_id == '' or pd.isna(doc_id):
                # For null document_id, we can insert it
                record['document_id'] = None  # Ensure it's actually NULL in the database
                null_id_records += 1
            elif doc_id in existing_doc_ids:
                # Skip if this document_id already exists
                skipped_records += 1
                continue
            
            # Only include fields that match our Document model
            valid_fields = {k: v for k, v in record.items() if hasattr(Document, k)}
            doc = Document(**valid_fields)
            session.add(doc)
            session.flush()  # Flush to catch any integrity errors
            
            # Add to existing IDs if not null
            if doc_id is not None and doc_id != '' and not pd.isna(doc_id):
                existing_doc_ids.add(doc_id)
                
            new_records += 1
            
        except Exception as inner_e:
            session.rollback()
            print(f"Error adding record: {inner_e}")
            print(f"Problematic record: {record}")
            skipped_records += 1
    
    # Commit the changes after all additions
    session.commit()
    print(f"Added {new_records} new records to the database.")
    print(f"  - {null_id_records} records with NULL document_id")
    print(f"  - {new_records - null_id_records} records with valid document_id")
    print(f"Skipped {skipped_records} records (already exist or had errors).")
    
except Exception as e:
    session.rollback()
    print(f"Error during database insertion: {e}")
    raise
finally:
    session.close()

print("\nDatabase setup complete!")


Added 369 new records to the database.
  - 369 records with NULL document_id
  - 0 records with valid document_id
Skipped 62 records (already exist or had errors).

Database setup complete!


## Comments, dockets, and grades

In [67]:
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, Boolean, ForeignKey, DateTime, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
import psycopg2
import re
from datetime import datetime

# Database connection parameters 
username = "postgres"
password = "your_password"
host = "localhost"
port = "5432"
database_name = "ycej_database"

# Create database engine
engine = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{database_name}')

# Create a base class for declarative class definitions
Base = declarative_base()

/var/folders/2z/q_7y7r7525qcys0tvfht24m00000gp/T/ipykernel_63515/2880386559.py:21: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [68]:

# Add new models for Dockets, Comments, and Feedback
class Docket(Base):
    __tablename__ = 'dockets'
    
    docket_id = Column(String, primary_key=True)
    docket_type = Column(String)
    last_modified_date = Column(DateTime)
    agency_id = Column(String)
    title = Column(String)
    object_id = Column(String)
    
    # Relationship to comments
    # This will create a one-to-many relationship
    comments = relationship("Comment", back_populates="docket")
    
    def __repr__(self):
        return f"<Docket(docket_id='{self.docket_id}', title='{self.title}')>"

class Comment(Base):
    __tablename__ = 'comments'
    
    comment_id = Column(String, primary_key=True)
    docket_id = Column(String, ForeignKey('dockets.docket_id'))
    comment_on_document_id = Column(String)
    comment_text = Column(Text)  # Will combine from EPA_Comments and extracted_texts
    posted_date = Column(DateTime)
    object_id = Column(String)
    duplicate_comments = Column(Integer)
    document_type = Column(String)
    
    # Additional columns you might want to keep
    tracking_nbr = Column(String)
    submitter_rep = Column(String)
    
    # Relationships
    docket = relationship("Docket", back_populates="comments")
    feedback = relationship("Feedback", back_populates="comment", uselist=False)
    
    def __repr__(self):
        return f"<Comment(comment_id='{self.comment_id}', docket_id='{self.docket_id}')>"

class Feedback(Base):
    __tablename__ = 'feedback'
    
    id = Column(Integer, primary_key=True)
    comment_id = Column(String, ForeignKey('comments.comment_id'))
    graded_feedback = Column(Text)
    
    # Relationship
    comment = relationship("Comment", back_populates="feedback")
    
    def __repr__(self):
        return f"<Feedback(id={self.id}, comment_id='{self.comment_id}')>"

# Create all tables in the database
Base.metadata.create_all(engine)

In [69]:

# 1. Load EPA Headers
print("Loading EPA Headers data...")
epa_headers = pd.read_csv('EPA_All_Headers_Up_To_2025-02-11.csv')
epa_headers.columns = [col.lower() for col in epa_headers.columns]  # Convert column names to lowercase

# 2. Load EPA Comments
print("Loading EPA Comments data...")
epa_comments = pd.read_csv('EPA_Comments.csv')
epa_comments.columns = [col.lower() for col in epa_comments.columns]  # Convert column names to lowercase

# 3. Load Extracted Texts
print("Loading Extracted Texts data...")
try:
    extracted_texts = pd.read_csv('extracted_texts.csv')
    extracted_texts.columns = [col.lower() for col in extracted_texts.columns]
except Exception as e:
    print(f"Error loading extracted_texts.csv: {e}")
    extracted_texts = pd.DataFrame(columns=['commentid', 'docketid', 'pdf_text'])

# 4. Load Graded Feedback
print("Loading Graded Feedback data...")
feedback_data = pd.read_csv('graded_feedback.csv')
feedback_data.columns = [col.lower() for col in feedback_data.columns]


Loading EPA Headers data...
Loading EPA Comments data...
Loading Extracted Texts data...
Loading Graded Feedback data...


In [70]:
# Function to clean and prepare data for insertion
def clean_dataframe(df):
    # Handle date columns - convert string dates to datetime
    date_columns = [col for col in df.columns if 'date' in col.lower()]
    for col in date_columns:
        if col in df.columns:
            try:
                df[col] = pd.to_datetime(df[col], errors='coerce')
            except:
                pass
    
    # Clean text columns - remove problematic characters if needed
    text_columns = df.select_dtypes(include=['object']).columns
    for col in text_columns:
        df[col] = df[col].fillna('')
    
    return df

# Clean datasets
epa_headers = clean_dataframe(epa_headers)
epa_comments = clean_dataframe(epa_comments)
if 'text' in extracted_texts.columns:
    extracted_texts = clean_dataframe(extracted_texts)
feedback_data = clean_dataframe(feedback_data)

# Combine EPA Comments with Extracted Texts to get complete comment texts
print("Merging comment data...")
# Using column names from extracted_texts dataframe
if not extracted_texts.empty and 'commentid' in extracted_texts.columns and 'pdf_text' in extracted_texts.columns:
    # Create a dictionary of extracted texts with commentid as the key and pdf_text as the value
    extracted_text_dict = dict(zip(extracted_texts['commentid'], extracted_texts['pdf_text']))
    
    # Function to get best comment text
    def get_best_comment_text(row):
        comment_id = row['commentid']
        # If comment is empty (or questionably short ?) in epa_comments, use extracted text if available
        if pd.isna(row['comment']) or str(row['comment']).strip() == '' or len(str(row['comment'])) < 10:
            return extracted_text_dict.get(comment_id, '')
        return row['comment']
    
    # Add a combined comment_text column
    epa_comments['comment_text'] = epa_comments.apply(get_best_comment_text, axis=1)
else:
    # If no extracted_texts with the right columns, just use the comment column
    print("Warning: extracted_texts file doesn't have the expected structure. Using original comment text only.")
    epa_comments['comment_text'] = epa_comments['comment']

Merging comment data...


In [72]:
# Create a session to interact with the database
Session = sessionmaker(bind=engine)
session = Session()


inserting into database!!!

In [73]:

try:
    # Insert dockets data
    print("Inserting dockets data...")
    dockets_inserted = 0
    dockets_skipped = 0
    
    # Get existing docket IDs
    existing_docket_ids = set(row[0] for row in session.query(Docket.docket_id).all())
    
    for _, row in epa_headers.iterrows():
        try:
            # Skip if docket already exists
            if row['docketid'] in existing_docket_ids:
                dockets_skipped += 1
                continue
                
            docket = Docket(
                docket_id=row['docketid'],
                docket_type=row.get('dockettype', ''),
                last_modified_date=row.get('lastmodifieddate'),
                agency_id=row.get('agencyid', ''),
                title=row.get('title', ''),
                object_id=row.get('objectid', '')
            )
            
            session.add(docket)
            session.flush()
            
            # Add to existing IDs set
            existing_docket_ids.add(row['docketid'])
            dockets_inserted += 1
            
            # Commit every 1000 records to avoid memory issues
            if dockets_inserted % 1000 == 0:
                session.commit()
                print(f"  {dockets_inserted} dockets inserted so far...")
        
        except Exception as e:
            session.rollback()
            print(f"Error inserting docket {row.get('docketid', 'unknown')}: {e}")
            dockets_skipped += 1
    
    # Final commit for dockets
    session.commit()
    print(f"Dockets insertion complete: {dockets_inserted} inserted, {dockets_skipped} skipped")
    
    # Insert comments data
    print("Inserting comments data...")
    comments_inserted = 0
    comments_skipped = 0
    
    # Get existing comment IDs
    existing_comment_ids = set(row[0] for row in session.query(Comment.comment_id).all())
    
    for _, row in epa_comments.iterrows():
        try:
            # Skip if comment already exists
            if row['commentid'] in existing_comment_ids:
                comments_skipped += 1
                continue
                
            comment = Comment(
                comment_id=row['commentid'],
                docket_id=row.get('docketid', ''),
                comment_on_document_id=row.get('commentondocumentid', ''),
                comment_text=row.get('comment_text', ''),
                posted_date=row.get('posteddate'),
                object_id=row.get('objectid', ''),
                duplicate_comments=row.get('duplicatecomments') if not pd.isna(row.get('duplicatecomments')) else None,
                document_type=row.get('documenttype', ''),
                tracking_nbr=row.get('trackingnbr', ''),
                submitter_rep=row.get('submitterrep', '')
            )
            
            session.add(comment)
            session.flush()
            
            # Add to existing IDs set
            existing_comment_ids.add(row['commentid'])
            comments_inserted += 1
            
            # Commit every 1000 records
            if comments_inserted % 1000 == 0:
                session.commit()
                print(f"  {comments_inserted} comments inserted so far...")
        
        except Exception as e:
            session.rollback()
            print(f"Error inserting comment {row.get('commentid', 'unknown')}: {e}")
            comments_skipped += 1
    
    # Final commit for comments
    session.commit()
    print(f"Comments insertion complete: {comments_inserted} inserted, {comments_skipped} skipped")
    
    # Insert feedback data
    print("Inserting feedback data...")
    feedback_inserted = 0
    feedback_skipped = 0
    
    for _, row in feedback_data.iterrows():
        try:
            # Check if the comment exists before adding feedback for it
            comment_exists = session.query(Comment).filter(Comment.comment_id == row['commentid']).first() is not None
            
            if not comment_exists:
                feedback_skipped += 1
                continue
                
            feedback = Feedback(
                comment_id=row['commentid'],
                graded_feedback=row.get('graded_feedback', '')
            )
            
            session.add(feedback)
            feedback_inserted += 1
            
            # Commit every 500 records
            if feedback_inserted % 500 == 0:
                session.commit()
        
        except Exception as e:
            session.rollback()
            print(f"Error inserting feedback for comment {row.get('commentid', 'unknown')}: {e}")
            feedback_skipped += 1
    
    # Final commit for feedback
    session.commit()
    print(f"Feedback insertion complete: {feedback_inserted} inserted, {feedback_skipped} skipped")
    
    print("\nDatabase setup complete!")
    
except Exception as e:
    session.rollback()
    print(f"Error during database setup: {e}")
    raise
finally:
    session.close()

Inserting dockets data...
  1000 dockets inserted so far...
  2000 dockets inserted so far...
  3000 dockets inserted so far...
  4000 dockets inserted so far...
  5000 dockets inserted so far...
  6000 dockets inserted so far...
  7000 dockets inserted so far...
  8000 dockets inserted so far...
  9000 dockets inserted so far...
  10000 dockets inserted so far...
  11000 dockets inserted so far...
  12000 dockets inserted so far...
  13000 dockets inserted so far...
  14000 dockets inserted so far...
  15000 dockets inserted so far...
  16000 dockets inserted so far...
  17000 dockets inserted so far...
  18000 dockets inserted so far...
  19000 dockets inserted so far...
  20000 dockets inserted so far...
Dockets insertion complete: 20682 inserted, 0 skipped
Inserting comments data...
Comments insertion complete: 393 inserted, 0 skipped
Inserting feedback data...
Feedback insertion complete: 109 inserted, 0 skipped

Database setup complete!
